# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

# Load & Inspect data

In [0]:
df_raw = spark.table("workspace.bronze.erp_cust_az12_raw")
df_raw.display()

In [0]:
print(df_raw.count())
df_raw.printSchema()

# Transform data

## Rename columns

In [0]:
df_raw.schema.fields

In [0]:
name_map = {
    'CID' : 'customer_key',
    'BDATE' : 'birth_date',
    'GEN' : 'gender'
}

df = df_raw

for f in df.schema.fields:
    df = df.withColumnRenamed(f.name, name_map[f.name])

df.display()

## Check null & distinct 

In [0]:
print(f'The table has {df.count()} rows')

id_count = df.select('customer_key').distinct().count()
print(f'There are {id_count} unque ids')

In [0]:
df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns
]).display()

## Align 'customer_key'

In [0]:
df = df.withColumn(
    'customer_key', 
    F.substring(F.col('customer_key'), 4, F.length(F.col('customer_key')))
)

df.display()

# Write table

In [0]:
df.write.mode("overwrite").saveAsTable("workspace.silver.erp_customers")

## Sanity check

In [0]:
%sql
SELECT * FROM workspace.silver.erp_customers